# cncf/sbom corpus: coverage, freshness, quality

Reads what `python -m sbom_corpus` builds under `output/sbom-corpus/`:
`bomhort_inventory.parquet` (every SBOM BOMHort holds), `inventory.parquet` (the public OCI bucket)
and `sbom.duckdb` (normalized sample). All queries run in DuckDB, so nothing has to fit in memory;
only the aggregated results reach Altair.

```bash
python -m sbom_corpus inventory && python -m sbom_corpus fetch
python -m sbom_corpus bomhort
python -m sbom_corpus load --collector-db site/public/data
```

In [1]:
from pathlib import Path
import altair as alt
import duckdb

DATA = Path('../output/sbom-corpus') if Path('../output/sbom-corpus').exists() else Path('output/sbom-corpus')
con = duckdb.connect(str(DATA / 'sbom.duckdb'), read_only=True)
con.execute(f"CREATE TEMP VIEW bomhort AS SELECT * FROM read_parquet('{DATA / 'bomhort_inventory.parquet'}')")
con.execute(f"CREATE TEMP VIEW public_bucket AS SELECT * FROM read_parquet('{DATA / 'inventory.parquet'}')")
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Freshness: SBOMs ingested by BOMHort per week, vs. the frozen public bucket

In [2]:
weekly = con.sql('''
    SELECT date_trunc('week', ingested_at)::DATE AS week, 'BOMHort (live)' AS source, count(*) AS sboms FROM bomhort GROUP BY ALL
    UNION ALL
    SELECT date_trunc('week', last_modified)::DATE, 'public OCI bucket', count(*) FROM public_bucket
    WHERE kind = 'spdx' AND NOT is_bomhort_copy GROUP BY ALL
''').df()
alt.Chart(weekly).mark_bar().encode(
    x=alt.X('week:T', title=None), y=alt.Y('sboms:Q', title='SBOMs', scale=alt.Scale(type='symlog')),
    color=alt.Color('source:N', title=None), tooltip=['week:T', 'source:N', 'sboms:Q'],
).properties(width=640, height=240, title='Weekly SBOM arrivals (log scale)')

alt.Chart(...)

## Coverage of the CNCF landscape (cncf/sbom's own project list)

In [3]:
cov = con.sql('''
    SELECT project_status AS maturity,
           count(DISTINCT project_slug) FILTER (WHERE project_slug IN (SELECT project FROM bomhort)) AS covered,
           count(DISTINCT project_slug) - covered AS missing
    FROM ref_sbom_repos WHERE source = 'cncf' GROUP BY 1
''').df().melt(id_vars='maturity', var_name='state', value_name='projects')
alt.Chart(cov).mark_bar().encode(
    y=alt.Y('maturity:N', sort=['graduated', 'incubating', 'sandbox'], title=None),
    x=alt.X('projects:Q', stack='zero'), color=alt.Color('state:N', scale=alt.Scale(range=['#4c78a8', '#e45756'])),
    tooltip=['maturity', 'state', 'projects'],
).properties(width=560, height=120, title='CNCF projects with at least one SBOM in BOMHort')

alt.Chart(...)

## Which generator made the newest SBOM of each release in the sample

In [4]:
gen = con.sql('''
    SELECT d.generator || coalesce(' ' || d.generator_version, '') AS generator, count(*) AS releases,
           avg(d.package_count) AS avg_packages
    FROM base_sbom_releases r JOIN base_sbom_documents d USING (sbom_id) GROUP BY 1 ORDER BY 2 DESC
''').df()
alt.Chart(gen).mark_bar().encode(
    x='releases:Q', y=alt.Y('generator:N', sort='-x', title=None), tooltip=['generator', 'releases', 'avg_packages'],
).properties(width=560, height=200, title='Preferred SBOM per release, by generator')

alt.Chart(...)

## Field fill rates by generator (non-root packages)

In [5]:
fill = con.sql('''
    SELECT d.generator,
           avg((p.purl IS NOT NULL)::INT) AS purl, avg((p.license_declared IS NOT NULL)::INT) AS license_declared,
           avg((p.license_concluded IS NOT NULL)::INT) AS license_concluded, avg(p.has_checksum::INT) AS checksum,
           avg((p.supplier IS NOT NULL)::INT) AS supplier, avg((p.source_owner_repo IS NOT NULL)::INT) AS source_repo
    FROM base_sbom_packages p JOIN base_sbom_documents d USING (sbom_id) WHERE NOT p.is_root GROUP BY 1
''').df().melt(id_vars='generator', var_name='field', value_name='fill')
alt.Chart(fill).mark_rect().encode(
    x=alt.X('field:N', title=None), y=alt.Y('generator:N', title=None),
    color=alt.Color('fill:Q', scale=alt.Scale(scheme='blues', domain=[0, 1]), legend=alt.Legend(format='%')),
    tooltip=['generator', 'field', alt.Tooltip('fill:Q', format='.0%')],
).properties(width=420, height=180, title='Share of packages with the field populated')

alt.Chart(...)

## Most depended-on CNCF-tracked repositories (newest SBOM per repo)

In [6]:
top = con.sql('''
    WITH latest AS (SELECT * FROM (SELECT *, row_number() OVER (PARTITION BY owner_repo ORDER BY created DESC NULLS LAST) rn
                    FROM base_sbom_releases) WHERE rn = 1),
         tracked AS (SELECT DISTINCT owner_repo FROM ref_sbom_repos)
    SELECT p.source_owner_repo AS repo, count(DISTINCT l.owner_repo) AS dependents
    FROM latest l JOIN base_sbom_packages p USING (sbom_id)
    WHERE p.source_owner_repo IN (SELECT owner_repo FROM tracked) AND p.source_owner_repo <> l.owner_repo
    GROUP BY 1 ORDER BY 2 DESC LIMIT 20
''').df()
alt.Chart(top).mark_bar().encode(
    x='dependents:Q', y=alt.Y('repo:N', sort='-x', title=None), tooltip=['repo', 'dependents'],
).properties(width=520, height=400, title='CNCF repos most depended on by other CNCF repos')

alt.Chart(...)